# **Training Deep Q-Learning Agent**
---

In this doc file, we will talk about the training implementation into the **src/train.py** file.
It is cut into three main parts that orchestrate the learning process.

We will begin with the `Training()` class, then the training loop functions, and finally the main execution flow.

## 1. Import Required Libraries

In [ ]:
import argparse
import sys
import numpy as np
import os
import gymnasium as gym
import torch
import imageio
import yaml
from pathlib import Path
from collections import deque

sys.path.insert(0, ".")
from src.env_utils import make_env, get_termination_reason
from src.logger import EpisodeLogger
from src.policies import RandomPolicy, HeuristicPolicy
from src.lunarAI import ANN, ReplayMemory, Agent

print("✓ All libraries imported successfully")

---

## 2. Configuration Setup

In [ ]:
# Load configuration file
cfg_path = "../config/training_config.yaml"

with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

print("Configuration loaded:")
for key, value in cfg.items():
    print(f"  {key}: {value}")

---

### **Training Class**

The `Training()` class initializes and manages the training environment. Its goal is to set up all necessary components:
- Environment creation
- Logging system
- Model paths and directories
- Hyperparameters (epsilon, decay rate, etc.)

`__init__()` initializes the training environment with all configuration parameters. It takes the config dictionary and config file path.

In [ ]:
class Training():

    def __init__(self, cfg, cfg_path):
        self.run_name = cfg.get("run_name", Path(cfg_path).stem)
        self.algo = cfg.get("algo", "random")
        self.n_ep = cfg['n_episodes']
        self.seed = cfg['seed']
        base_dir = cfg.get("log_dir", "logs")
        self.epsilon = cfg.get('epsilon_start', 1.0)
        self.epsilon_end = cfg.get('epsilon_end', 0.01)
        self.epsilon_decay = cfg.get('epsilon_decay', 0.995)
        self.max_time_step = cfg.get('max_time_steps', 1000)
        self.env = make_env(self.seed, render_mode="rgb_array")
        self.state_size = self.env.observation_space.shape[0]
        self.action_size = self.env.action_space.n
        self.run_folder = f"{base_dir}/train/{self.run_name}"

        # Define model path
        os.makedirs(f"{base_dir}/brain", exist_ok=True)
        os.makedirs(self.run_folder, exist_ok=True)
        self.model_path = f"{base_dir}/brain/{self.run_name}_model.pth"
        self.logger = EpisodeLogger(f"{self.run_folder}/{self.run_name}.csv", run_name=self.run_name, verbose=True)
        self.frames = []

Key attributes:
- **`run_name`**: Identifier for this training run
- **`algo`**: Algorithm type ("random", "heuristic", or "dqn")
- **`epsilon`**: Exploration probability for epsilon-greedy strategy
- **`state_size` / `action_size`**: Environment dimensions
- **`logger`**: Tracks episode statistics

In [ ]:
# Initialize training environment
train = Training(cfg, cfg_path)
print(f"✓ Training environment initialized")
print(f"  State size: {train.state_size}")
print(f"  Action size: {train.action_size}")
print(f"  Total episodes: {train.n_ep}")
print(f"  Algorithm: {train.algo}")

---

## 3. Agent Instantiation

The agent is initialized based on the algorithm type specified in the config. Three types are supported:
- **RandomPolicy**: Takes random actions
- **HeuristicPolicy**: Uses hand-crafted heuristics
- **Agent (DQN)**: Deep Q-Network for learning

In [ ]:
# Initialize agent based on algorithm
if train.algo == "random":
    agent = RandomPolicy(train.env.action_space, train.seed)
    print(f"✓ Random agent initialized")
elif train.algo == "heuristic":
    agent = HeuristicPolicy()
    print(f"✓ Heuristic agent initialized")
elif train.algo == "dqn":
    try:
        agent = Agent(train.state_size, train.action_size, cfg=cfg)
        if os.path.exists(train.model_path):
            agent = torch.load(train.model_path, weights_only=False)
            print(f"✓ DQN agent loaded from {train.model_path}")
        else:
            print(f"✓ New DQN agent initialized")
    except ImportError:
        print("✗ DQN Failed import.")
        sys.exit(1)
else:
    print(f"✗ Unknown algo : {train.algo}")
    sys.exit(1)

---

## 4. Training Loop Functions

The training process is split into two main functions:

### **time_step_loop()**

This function executes one complete episode. It:
1. Selects actions using the agent (epsilon-greedy for DQN)
2. Steps the environment
3. Stores experiences in replay memory (if DQN)
4. Accumulates rewards
5. Logs the episode when done

It takes: agent, epsilon (exploration rate), state, score, training object, and episode length.

In [ ]:
def time_step_loop(agent, epsilon, state, score, train, length):
    """Execute one episode timestep loop"""
    for _ in range(0, train.max_time_step):
        # Select action
        if hasattr(agent, "get_action"):
            action = agent.get_action(state, epsilon)
        else:
            action = agent.select_action(state)

        # Step environment
        next_state, reward, terminated, truncated, info = train.env.step(action)
        done = terminated or truncated

        # Store experience in replay memory
        if hasattr(agent, "step"):
            agent.step(state, action, reward, next_state, done)

        state = next_state
        score += reward
        length += 1

        if done:
            reason = get_termination_reason(next_state, terminated, truncated, info)
            train.logger.log_episode(score=score, length=length,
                               terminated=terminated, truncated=truncated,
                               reason=reason, extra={"epsilon": epsilon})
            break
    
    return score, length

### **learn_loop()**

This is the main training loop that runs across all episodes. It:
1. Iterates through episodes
2. Resets the environment with a seeded state
3. Executes the timestep loop for each episode
4. Decays epsilon after each episode
5. Checks if the agent has solved the environment (avg score ≥ 200)
6. Saves the trained model
7. Generates logs and plots

It takes: agent and training object.

In [ ]:
def learn_loop(agent, train):
    """Main training loop across all episodes"""
    scores_100_episodes = deque(maxlen=100)
    
    for episode in range(0, train.n_ep):
        # Reset environment
        state, _ = train.env.reset(seed=train.seed + episode)
        score = 0
        length = 0
        
        # Run episode timesteps
        score, length = time_step_loop(agent, train.epsilon, state, score, train, length)
        
        # Track scores and update epsilon
        scores_100_episodes.append(score)
        train.epsilon = max(train.epsilon_end, train.epsilon * train.epsilon_decay)
        
        # Print progress every 10 episodes
        if episode % 10 == 0:
            print('Episode {} Avg Score: {:.2f}'.format(episode, np.mean(scores_100_episodes)))
        
        # Check if solved (average score >= 200 over 100 episodes)
        if np.mean(scores_100_episodes) >= 200:
            print('Congratulation, Solved in {:d} episodes \t Avg Score {:.2f}'.format(episode, np.mean(scores_100_episodes)))
            break

    # Save model and results
    if train.algo == "dqn":
        torch.save(agent, train.model_path)
        print(f"✓ Model saved to {train.model_path}")
    
    train.env.close()
    train.logger.print_summary()
    train.logger.generate_plots()
    print(f"✓ Done. CSV/Plots → {train.run_folder}/{train.run_name}.csv")
    
    return 0

---

## 5. Execute Training

In [ ]:
# Start training
print(f"\n{'='*50}")
print(f"Starting training with {train.algo} algorithm")
print(f"Run name: {train.run_name}")
print(f"Episodes: {train.n_ep}")
print(f"{'='*50}\n")

result = learn_loop(agent, train)
print(f"\nTraining completed with result code: {result}")

---

## Summary

The training pipeline consists of:

- **`Training()`**: Manages environment, logging, and hyperparameters
- **`time_step_loop()`**: Executes a single episode
- **`learn_loop()`**: Orchestrates the complete training across all episodes

The key features:
- Supports multiple algorithms (Random, Heuristic, DQN)
- Epsilon-greedy exploration with decay
- Episode logging and CSV export
- Model saving and loading
- Early stopping when environment is solved (avg score ≥ 200)